# QE Namelist Reference Dictionaries — Interactive Demo

This notebook demonstrates four namelist reference modules:

| Module | Codes |
|---|---|
| `pw_namelists.py` | `pw.x` |
| `ph_namelists.py` | `ph.x` · `q2r.x` · `matdyn.x` |
| `postproc_namelists.py` | `pp.x` · `bands.x` |
| `pdos_dos_namelists.py` | `projwfc.x` · `dos.x` |

It runs on **Google Colab** using a pre-built `qe_env` conda environment restored from Google Drive.

---
## ⚙️ Step 1 — Configuration  *(edit only this cell)*

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit this cell only
# ══════════════════════════════════════════════════════════════════════════════

# Option 1 — your own Google Drive path (default)
ENV_ARCHIVE = '/content/drive/MyDrive/conda_envs/qe_env.tar.gz'

# Option 2 — shared Google Drive folder
# ENV_ARCHIVE = '/content/drive/Shareddrives/QE_Tutorials/qe_env.tar.gz'

# Option 3 — shared via a Google Drive sharing link
# To use this:
#   1. The owner shares the file with 'Anyone with the link'
#   2. Copy the file ID from the URL:
#      https://drive.google.com/file/d/FILE_ID_HERE/view
#   3. Paste the FILE_ID below and uncomment these two lines:
# GDRIVE_FILE_ID = 'FILE_ID_HERE'
# ENV_ARCHIVE    = f'/content/qe_env.tar.gz'

# ══════════════════════════════════════════════════════════════════════════════
print(f'ENV_ARCHIVE set to: {ENV_ARCHIVE}')

## 🐍 Step 2 — Bootstrap condacolab

> **Note:** This cell triggers a **kernel restart** on first run.  
> After the restart, **re-run this cell once** — it will skip the install and continue.

In [ ]:
# ── Bootstrap condacolab (triggers kernel restart on first run) ───────────────
try:
    import condacolab
    condacolab.check()
    print('✅ condacolab active — continuing to restore …')
except Exception:
    import subprocess, sys
    print('Installing condacolab …')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'],
        stdout=subprocess.DEVNULL
    )
    import condacolab
    condacolab.install()    # ← kernel restarts here; re-run this cell after restart

## 📦 Step 3 — Restore `qe_env`, expose packages, install ovito

In [ ]:
# ── Restore qe_env from Drive ─────────────────────────────────────────────────
import subprocess, os, sys, glob

ENV_PATH = '/usr/local/envs/qe_env'

# Option 3: download from a sharing link with gdown
if 'GDRIVE_FILE_ID' in dir() and not os.path.isfile(ENV_ARCHIVE):
    print('Downloading from Google Drive sharing link …')
    subprocess.check_call(
        ['pip', 'install', '-q', 'gdown'], stdout=subprocess.DEVNULL
    )
    import gdown
    gdown.download(id=GDRIVE_FILE_ID, output=ENV_ARCHIVE, quiet=False)

# Options 1 & 2: mount Drive
elif ENV_ARCHIVE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Restore the environment
if not os.path.isdir(ENV_PATH):
    print(f'Restoring qe_env from {ENV_ARCHIVE} …')
    os.makedirs(ENV_PATH, exist_ok=True)
    subprocess.run(
        ['tar', '-xzf', ENV_ARCHIVE, '-C', ENV_PATH],
        check=True
    )
    print('✅ Environment restored.')
else:
    print('✅ qe_env already present on disk.')

# ── Expose Python packages to this interpreter ────────────────────────────────
for sp in glob.glob('/usr/local/envs/qe_env/lib/python*/site-packages'):
    if sp not in sys.path:
        sys.path.insert(0, sp)
        print(f'✅ Added to sys.path: {sp}')

# ── Install ovito via pip (conda version incompatible with Colab Python 3.12) ─
print('Installing ovito via pip …')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'ovito'],
    stdout=subprocess.DEVNULL
)

# ── Verify ────────────────────────────────────────────────────────────────────
print('\nPackage availability:')
for pkg in ['numpy', 'matplotlib', 'ase', 'ovito']:
    try:
        __import__(pkg)
        print(f'  ✅  {pkg}')
    except ImportError as e:
        print(f'  ❌  {pkg} — {e}')

print('\nQE executables:')
for exe in ['pw.x', 'ph.x', 'pp.x', 'bands.x', 'dos.x', 'projwfc.x']:
    found = subprocess.run(
        ['conda', 'run', '-n', 'qe_env', 'which', exe],
        capture_output=True, text=True
    )
    status = '✅' if found.returncode == 0 else '❌'
    print(f'  {status}  {exe:14s}  {found.stdout.strip()}')

print('\n🎉 Ready — run QE with: !conda run -n qe_env pw.x < input.in')

## 📥 Step 4 — Download the namelist modules and pseudopotentials

In [ ]:
import subprocess, os

# ── Download namelist reference modules from GitHub (adjust URL as needed) ───
# Replace the URL below with wherever you host the three .py files.
# If running locally, just place them alongside this notebook instead.
BASE_URL = 'https://raw.githubusercontent.com/YOUR_USER/YOUR_REPO/main/'

for fname in ['pw_namelists.py', 'ph_namelists.py', 'postproc_namelists.py', 'pdos_dos_namelists.py']:
    if not os.path.isfile(fname):
        subprocess.run(['wget', '-q', BASE_URL + fname], check=True)
        print(f'✅ Downloaded {fname}')
    else:
        print(f'✅ {fname} already present')

# ── Pseudopotentials ──────────────────────────────────────────────────────────
PSEUDO_DIR = '/content/pseudo'
os.makedirs(PSEUDO_DIR, exist_ok=True)

pseudos = {
    'Si.pbe-n-kjpaw_psl.1.0.0.UPF':
        'https://pseudopotentials.quantum-espresso.org/upf_files/Si.pbe-n-kjpaw_psl.1.0.0.UPF',
}

for fname, url in pseudos.items():
    dest = os.path.join(PSEUDO_DIR, fname)
    if not os.path.isfile(dest):
        subprocess.run(['wget', '-q', '-O', dest, url], check=True)
        print(f'✅ Downloaded {fname}')
    else:
        print(f'✅ {fname} already present')

---
## 📖 Namelist reference — exploring parameters

The modules expose each namelist as a plain Python dict.  Every entry has the
same schema: `default`, `type`, `unit`, `description`, `valid`.

In [ ]:
from pw_namelists import (
    CONTROL, SYSTEM, ELECTRONS, IONS, CELL,
    PW_NAMELISTS, PW_CARDS,
    describe as pw_describe,
)
from postproc_namelists import (
    PP_INPUTPP, PP_PLOT,
    BANDS_INPUTBANDS,
    describe,
)
from ph_namelists import (
    PH_INPUTPH,
    Q2R_INPUT,
    MATDYN_INPUT,
)
from pdos_dos_namelists import (
    PROJWFC_PROJWFC,
    DOS_DOS,
    defaults_from,
)

print('Modules loaded ✅')

In [ ]:
# Human-readable summary of any parameter
pw_describe(SYSTEM, 'ecutwfc')

In [ ]:
# Access fields directly
print('type   :', SYSTEM['ecutwfc']['type'])
print('unit   :', SYSTEM['ecutwfc']['unit'])
print('default:', SYSTEM['ecutwfc']['default'])

In [ ]:
# List all parameters in a namelist
print('ELECTRONS parameters:')
for k, v in ELECTRONS.items():
    unit = f" [{v['unit']}]" if v.get('unit') else ''
    print(f"  {k:<30s}  default={str(v['default']):<14s}{unit}")

In [ ]:
# Tabular view with pandas
import pandas as pd

def namelist_to_df(namelist: dict) -> pd.DataFrame:
    rows = [
        {'parameter': k,
         'type': v.get('type', ''),
         'unit': v.get('unit', ''),
         'default': v.get('default'),
         'description': v.get('description', '')}
        for k, v in namelist.items()
    ]
    return pd.DataFrame(rows).set_index('parameter')

namelist_to_df(SYSTEM)

---
## 🏗️ Building pw.x inputs from the dictionaries

In [ ]:
def write_namelist_block(name: str, params: dict) -> str:
    """Format a Fortran namelist block as a string."""
    lines = [f'&{name}']
    for k, v in params.items():
        if isinstance(v, bool):
            v = '.true.' if v else '.false.'
        elif isinstance(v, str):
            v = f"'{v}'"
        lines.append(f'  {k} = {v}')
    lines.append('/')
    return '\n'.join(lines)

def validate_param(namelist: dict, key: str, value) -> bool:
    """Return True if value is a valid choice for the given parameter."""
    spec = namelist.get(key)
    if spec is None:
        print(f'  WARNING: {key!r} not found in namelist')
        return False
    valid = spec.get('valid', [])
    if valid and value not in valid:
        print(f'  ERROR: {key!r} = {value!r} not in {valid}')
        return False
    return True

print('Helpers defined ✅')

In [ ]:
# Build a silicon SCF input
scf_control = {
    **defaults_from(CONTROL),
    'calculation': 'scf',
    'prefix': 'silicon',
    'pseudo_dir': '/content/pseudo',
    'outdir': '/content/out',
    'tprnfor': True,
    'tstress': True,
}

scf_system = {
    'ibrav': 2,
    'celldm(1)': 10.26,
    'nat': 2,
    'ntyp': 1,
    'ecutwfc': 40.0,
    'ecutrho': 320.0,
    'occupations': 'smearing',
    'smearing': 'methfessel-paxton',
    'degauss': 0.02,
}

scf_electrons = defaults_from(ELECTRONS)

scf_in = (
    write_namelist_block('CONTROL', scf_control) + '\n\n' +
    write_namelist_block('SYSTEM',  scf_system)  + '\n\n' +
    write_namelist_block('ELECTRONS', scf_electrons) + '\n\n' +
    'ATOMIC_SPECIES\n  Si  28.0855  Si.pbe-n-kjpaw_psl.1.0.0.UPF\n\n' +
    'ATOMIC_POSITIONS {alat}\n  Si  0.00  0.00  0.00\n  Si  0.25  0.25  0.25\n\n' +
    'K_POINTS {automatic}\n  8 8 8  0 0 0\n'
)

# Write to file
import os
os.makedirs('/content/out', exist_ok=True)
with open('/content/scf.in', 'w') as f:
    f.write(scf_in)

print(scf_in)

---
## ▶️ Running pw.x SCF

In [ ]:
import subprocess

result = subprocess.run(
    ['conda', 'run', '-n', 'qe_env', 'pw.x', '-input', '/content/scf.in'],
    capture_output=True, text=True
)

# Print last 40 lines of output
print('\n'.join(result.stdout.splitlines()[-40:]))
if result.returncode != 0:
    print('\n--- STDERR ---')
    print(result.stderr[-2000:])

---
## 📊 DOS example — building dos.x input from the dictionary

In [ ]:
# First we need an nscf calculation on a denser k-mesh
nscf_control = {**scf_control, 'calculation': 'nscf', 'verbosity': 'high'}
nscf_system  = {**scf_system, 'nbnd': 12, 'occupations': 'tetrahedra'}
del nscf_system['smearing'], nscf_system['degauss']

nscf_in = (
    write_namelist_block('CONTROL',   nscf_control) + '\n\n' +
    write_namelist_block('SYSTEM',    nscf_system)  + '\n\n' +
    write_namelist_block('ELECTRONS', defaults_from(ELECTRONS)) + '\n\n' +
    'ATOMIC_SPECIES\n  Si  28.0855  Si.pbe-n-kjpaw_psl.1.0.0.UPF\n\n' +
    'ATOMIC_POSITIONS {alat}\n  Si  0.00  0.00  0.00\n  Si  0.25  0.25  0.25\n\n' +
    'K_POINTS {automatic}\n  16 16 16  0 0 0\n'
)
with open('/content/nscf.in', 'w') as f:
    f.write(nscf_in)

# --- DOS namelist from dictionary ---
dos_params = {
    **defaults_from(DOS_DOS),
    'prefix': 'silicon',
    'outdir': '/content/out',
    'fildos': '/content/silicon.dos',
    'Emin': -15.0,
    'Emax':  15.0,
    'DeltaE': 0.05,
}
dos_in = write_namelist_block('DOS', dos_params)
with open('/content/dos.in', 'w') as f:
    f.write(dos_in)

print(dos_in)

In [ ]:
# Describe any DOS parameter interactively
describe(DOS_DOS, 'ngauss')

In [ ]:
# Validate before running
for key, val in dos_params.items():
    ok = validate_param(DOS_DOS, key, val)
    if ok and DOS_DOS.get(key, {}).get('valid'):
        print(f'  ✅  {key} = {val}')

---
## 🔬 PDOS example — projwfc.x namelist from the dictionary

In [ ]:
projwfc_params = {
    **defaults_from(PROJWFC_PROJWFC),
    'prefix':  'silicon',
    'outdir':  '/content/out',
    'filpdos': '/content/silicon',
    'Emin':   -15.0,
    'Emax':    15.0,
    'DeltaE':   0.05,
    'ngauss':   0,
    'degauss':  0.01,
}
projwfc_in = write_namelist_block('PROJWFC', projwfc_params)
with open('/content/projwfc.in', 'w') as f:
    f.write(projwfc_in)

print(projwfc_in)

In [ ]:
# Explore projwfc.x output file naming convention
describe(PROJWFC_PROJWFC, 'filpdos')

---
## 🎸 ph.x + q2r.x + matdyn.x — phonon workflow templates

In [ ]:
ph_params = {
    **defaults_from(PH_INPUTPH),
    'prefix':  'silicon',
    'outdir':  '/content/out',
    'fildyn':  '/content/si.dyn',
    'ldisp':   True,
    'nq1': 4, 'nq2': 4, 'nq3': 4,
    'tr2_ph':  1e-12,
    'epsil':   True,
}
with open('/content/ph.in', 'w') as f:
    f.write(write_namelist_block('INPUTPH', ph_params))
print(write_namelist_block('INPUTPH', ph_params))

In [ ]:
q2r_params = {
    'fildyn': '/content/si.dyn',
    'flfrc':  '/content/si.fc',
    'zasr':   'crystal',
}
# Show valid ASR choices from the dict
print('Valid zasr options:', Q2R_INPUT['zasr']['valid'])
print()
print(write_namelist_block('INPUT', q2r_params))

In [ ]:
matdyn_params = {
    'flfrc':          '/content/si.fc',
    'asr':            'crystal',
    'flfrq':          '/content/si.freq',
    'flvec':          '/content/si.modes',
    'q_in_band_form': True,
    'q_in_cryst_coord': True,
}
print(write_namelist_block('INPUT', matdyn_params))

---
## 📋 Full reference tables for all codes

In [ ]:
from postproc_namelists import PP_NAMELISTS, BANDS_NAMELISTS
from ph_namelists import PH_NAMELISTS, Q2R_NAMELISTS, MATDYN_NAMELISTS
from pdos_dos_namelists import PROJWFC_NAMELISTS, DOS_NAMELISTS

all_codes = {
    'pw.x':       PW_NAMELISTS,
    'pp.x':       PP_NAMELISTS,
    'bands.x':    BANDS_NAMELISTS,
    'ph.x':       PH_NAMELISTS,
    'q2r.x':      Q2R_NAMELISTS,
    'matdyn.x':   MATDYN_NAMELISTS,
    'projwfc.x':  PROJWFC_NAMELISTS,
    'dos.x':      DOS_NAMELISTS,
}

for code, namelists in all_codes.items():
    for nl_name, nl in namelists.items():
        print(f'{code}  &{nl_name}  → {len(nl)} parameters')

In [ ]:
# Interactive lookup: choose code and parameter
code   = 'dos.x'
param  = 'DeltaE'

nl = list(all_codes[code].values())[0]   # first (often only) namelist
if param in nl:
    describe(nl, param)
else:
    print(f'{param!r} not found in {code}. Available: {list(nl)}')